# You do not need the workstation

**Reproducing this project's headline model on a student Colab subscription.**

CSED 505 · week 3, *infrastructure that survives you* · [WashingtonCsed504](https://github.com/TrueRottweiler/WashingtonCsed504)

**Use an A100.** Budget about 90 minutes, comfortably inside a Colab Pro session.

## Two questions, one run

The bottom poster reports 105 trained models on a workstation holding two RTX PRO 6000
cards. That is roughly **$24,000 of graphics cards** before the machine around them, and it
is the kind of number that makes a reader stop reading, because the implicit message is
*this work is not available to you.*

It is. This notebook rebuilds the corpus and retrains the exact headline model here, and the
last cell works out what the **whole 83-GPU-hour project** would cost at Colab prices.

And while it runs, it settles a second question we asserted and never measured.

## The second question

When Patrick asked for a checkpoint rather than retraining it locally, his reasoning was that
*same corpus, same seed, same steps on a different GPU is a different model* — and that under one
tag, nothing would say so. We agreed, wrote it into the record-keeping rules, and **never measured
it.** This measures it.

The workstation produced `yor_64M_62.5k_s0` at **validation loss 2.315** in 40.2 minutes. This
notebook runs the identical recipe — same corpus, same 64M tokens, same 62,500 steps, same seed 0,
same learning rate — on Colab's hardware, and compares.

## Why it cannot fail to produce a finding

There are two ways it can go and both are worth printing:

1. **The vocabulary comes back different.** `prepare_corpus` retrains the BPE from a FineWeb-2
   stream, and if the stream yields different text the fingerprint will not match
   `15abd33de5af`. Then the two runs are not comparable *at all* — which is precisely the trap
   the fingerprint system exists to catch, demonstrated live rather than asserted.
2. **The vocabulary matches and the loss does not.** Then we can finally say how big "a different
   model" actually is, in the same units as the seed spread — and decide whether Patrick's
   caution was worth a 125 MB upload.

**Keep this tab open.** Colab disconnects idle sessions. Everything is written incrementally and
`reuse=True`, so a reconnect resumes rather than restarts.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
%pip install --quiet transformers datasets tokenizers

!git clone --quiet --depth 1 https://github.com/TrueRottweiler/WashingtonCsed504.git
%cd WashingtonCsed504/src/a2-nlp
print('ready')

In [ ]:
# Prepare the Yoruba corpus from FineWeb-2. data/ is not in the repository -- 153 MB of token
# array is not something to keep in git -- so this rebuilds it, which is the part of the
# experiment that tests whether "the same corpus" is even reproducible. Five to ten minutes.
import mlm_api as factory

REFERENCE = {'tag': 'yor_64M_62.5k_s0', 'val_loss': 2.315156, 'minutes': 40.2,
             'fingerprint': '15abd33de5af', 'card': 'RTX PRO 6000 Blackwell Max-Q'}

stats = factory.prepare_corpus('yor', lang='yor_Latn')
fp = stats.get('tokenizer_fingerprint')

print(f"\nvocabulary fingerprint here : {fp}")
print(f"on the workstation          : {REFERENCE['fingerprint']}")
MATCH = fp == REFERENCE['fingerprint']
print('\nMATCH -- the two runs will be comparable.' if MATCH else
      '\nDIFFERENT. The corpus rebuilt to a different vocabulary, so the losses below are NOT\n'
      'comparable to 2.315 and must not be quoted against it. That is the finding: "same corpus"\n'
      'is not reproducible from a streaming source, and this is what the fingerprint is for.')

In [ ]:
# The run. Identical arguments to the workstation's, which is the whole point -- if anything here
# is edited, the comparison stops meaning anything.
import time

t0 = time.time()
rec = factory.pretrain('yor', tokens=64_000_000, steps=62_500, seed=0,
                       preset='poc', lr=5e-4, tag='colab_yor_64M_62.5k_s0')
print(f"\nfinished in {(time.time()-t0)/60:.1f} min")

In [ ]:
# The comparison, and the block to send back.
import json, platform

import torch

delta = rec['val_loss'] - REFERENCE['val_loss']
# The yardstick that matters: this project's rule is that a difference smaller than the cell's own
# seed spread is not a difference. For the 33.8M Yoruba cell at three seeds that spread is 0.103.
SEED_SPREAD = 0.103

out = {
    'label': 'Colab A100',
    'gpu': torch.cuda.get_device_name(0),
    'torch': torch.__version__,
    'platform': platform.platform(),
    'vocab_fingerprint_matched': MATCH,
    'val_loss_here': round(rec['val_loss'], 6),
    'val_loss_workstation': REFERENCE['val_loss'],
    'delta': round(delta, 6),
    'delta_in_seed_spreads': round(delta / SEED_SPREAD, 2),
    'minutes_here': round(rec['seconds'] / 60, 1),
    'minutes_workstation': REFERENCE['minutes'],
    'tokens_per_s': round(rec['tokens_per_s']),
}

print('-' * 68)
print(json.dumps(out, indent=2))
print('-' * 68)
if not MATCH:
    print('\nThe vocabularies differ, so the loss comparison above is meaningless. Report the\n'
          'fingerprint mismatch as the result and ignore the delta.')
elif abs(delta) < SEED_SPREAD:
    print(f'\n|delta| = {abs(delta):.3f} is INSIDE the {SEED_SPREAD} seed spread. Same recipe on\n'
          'different silicon lands within the noise the cell already has -- which would mean the\n'
          'hardware is not a source of difference worth worrying about at this scale.')
else:
    print(f'\n|delta| = {abs(delta):.3f} EXCEEDS the {SEED_SPREAD} seed spread. Hardware alone moved\n'
          'the result by more than reseeding does, which makes the checkpoint -- not the recipe --\n'
          'the thing that has to be shared. Patrick was right, and now we know by how much.')

In [ ]:
# What the whole project would cost here instead of on the workstation.
#
# Every rate below is somebody else's pricing page and they all change. They are constants at the
# top of this cell so a reader can correct them, rather than trusting a number baked into a
# poster -- the same discipline as the staleness check in the main notebook, pointed at Google.
PROJECT_GPU_HOURS = 83.3          # what A2 consumed, measured from our own run records
BUDGET_USD = 500                  # the question: does the project fit inside this?
UNITS_PER_HOUR = 11.8             # Colab A100 -- CHECK, this changes
USD_PER_100_UNITS = 9.99          # pay-as-you-go compute units -- CHECK
WORKSTATION_CARDS_USD = 24_000    # two RTX PRO 6000 Blackwell Max-Q

# Our sustained medians, 96 and 55 completed runs. The project is a mix of both model sizes, so
# scaling by only the small one would flatter the answer.
OURS = {'poc': 381_817, 'afriberta': 184_329}
SHARE = {'poc': 0.55, 'afriberta': 0.45}          # roughly how the 83 hours split

here = rec['tokens_per_s']                         # measured above, on the 33.8M model
ratio_small = OURS['poc'] / here
# The larger model is ~2.07x the cost per token on every card we have measured, so assume the
# ratio carries. Stated rather than hidden, because it is the one unmeasured step here.
ratio_big = ratio_small

hours = PROJECT_GPU_HOURS * (SHARE['poc'] * ratio_small + SHARE['afriberta'] * ratio_big)
units = hours * UNITS_PER_HOUR
cost = units / 100 * USD_PER_100_UNITS

print(f'this GPU runs the small model at {here:,.0f} tok/s, '
      f'{ratio_small:.2f}x the workstation')
print(f'the whole project here: {hours:.0f} GPU-hours '
      f'~= {units:,.0f} compute units ~= ${cost:,.0f}')
print()
if cost <= BUDGET_USD:
    print(f'FITS in ${BUDGET_USD}. You could reproduce all 105 models for '
          f'{cost/BUDGET_USD:.0%} of that budget,')
    print(f'against ${WORKSTATION_CARDS_USD:,} of cards -- '
          f'{WORKSTATION_CARDS_USD/max(cost,1):,.0f}x cheaper.')
else:
    print(f'DOES NOT fit in ${BUDGET_USD}; it is ${cost:,.0f}. '
          f'${BUDGET_USD} buys {BUDGET_USD/max(cost,1):.0%} of the project,')
    print('which is still every experiment that matters if you drop the seed replication.')
print()
print('The caveats, because this is the number people will quote:')
print('  - a subscription buys a QUEUE, not a machine. Sessions end, and the 34-hour studies')
print('    in this project would have to be cut into resumable pieces.')
print('  - you get whichever GPU is free, so a study split across an A100 and an L4 has')
print('    hardware as an uncontrolled variable -- which is what the fingerprint check above')
print('    is a demonstration of.')
print('  - owning still wins eventually: the crossover against rental is ~9,300 GPU-hours.')
print('    This project used 83, which is 0.9% of the way there.')


## If the session drops

Re-run every cell. `prepare_corpus` and `pretrain` both check for completed work first, so a
reconnect picks up where it stopped instead of paying for it twice — the same property that makes
the notebooks on the workstation cheap to re-run.

If the runtime is reassigned to a *different* GPU on reconnect, say so when you send the numbers.
Half a run on an A100 and half on an L4 is a third condition, not either of the two.